## 1. Load Original Data

We begin with the two original datasets:

- `pipes.csv`: contains information about all pipes in the network.
- `train.csv`: despite its name, this dataset contains historical leak events and their associated costs. We treat this as the `costs` dataset.

These two datasets will be used to construct the pipe-year master dataset for leak prediction.

In [9]:
import pandas as pd
import numpy as np

# Load original datasets
pipes = pd.read_csv("../data/pipes.csv")
costs = pd.read_csv("../data/train.csv")

print("pipes shape:", pipes.shape)
print("costs shape:", costs.shape)

pipes shape: (43039, 8)
costs shape: (14113, 4)


In [10]:
print("PIPES")
display(pipes.head())
print(pipes.dtypes)

print("\nCOSTS")
display(costs.head())
print(costs.dtypes)

PIPES


,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2
0,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97
1,P01002,1971-09-19,cast iron,water,6512.35,4147.58,6536.59,4201.94
2,P01003,1971-09-18,cast iron,water,6536.59,4201.94,6518.42,4201.93
3,P01004,1982-05-22,cast iron,structure,2836.68,3377.95,2848.80,3405.12
4,P01005,1971-07-03,cast iron,water,5117.28,4272.91,5126.37,4297.16


Pipe ID      object
Lay date     object
Material     object
Surface      object
GPS x1      float64
GPS y1      float64
GPS x2      float64
GPS y2      float64
dtype: object

COSTS


,Pipe ID,Date,Time,Cost
0,P19711,2019-01-06,08:20,2074.90
1,P07628,2019-01-07,00:00,618.71
2,P31195,2019-04-16,03:48,1050.07
3,P15715,2019-04-16,16:51,1000.81
4,P10415,2019-04-17,22:08,3157.83


Pipe ID     object
Date        object
Time        object
Cost       float64
dtype: object


## 2. Check Data Quality and Pipe IDs

Before constructing the pipe-year dataset, we check:

- Missing values in both datasets
- Whether each Pipe ID is unique in `pipes`
- Whether a pipe can appear multiple times in the historical leak data
- Whether every leak record can be matched to a pipe in `pipes`

These checks are important before defining the first-leak observation structure.

In [11]:
# Check dataset sizes
print("Number of pipes:", len(pipes))
print("Number of leak records:", len(costs))

# Check missing values
print("\nMissing values in pipes:")
display(pipes.isna().sum())

print("\nMissing values in costs:")
display(costs.isna().sum())

Number of pipes: 43039
Number of leak records: 14113

Missing values in pipes:


Pipe ID      0
Lay date    15
Material     8
Surface      0
GPS x1       0
GPS y1       0
GPS x2       0
GPS y2       0
dtype: int64


Missing values in costs:


Pipe ID    0
Date       0
Time       0
Cost       0
dtype: int64

In [12]:
# Check Pipe ID uniqueness
print("Unique Pipe IDs in pipes:", pipes["Pipe ID"].nunique())
print("Total rows in pipes:", len(pipes))

print("\nUnique Pipe IDs in costs:", costs["Pipe ID"].nunique())
print("Total rows in costs:", len(costs))

print("\nDuplicate Pipe IDs in pipes:",
      pipes["Pipe ID"].duplicated().sum())

print("Duplicate Pipe IDs in costs:",
      costs["Pipe ID"].duplicated().sum())

Unique Pipe IDs in pipes: 43039
Total rows in pipes: 43039

Unique Pipe IDs in costs: 14113
Total rows in costs: 14113

Duplicate Pipe IDs in pipes: 0
Duplicate Pipe IDs in costs: 0


In [13]:
# Check whether every leak Pipe ID exists in pipes
unmatched = costs.loc[
    ~costs["Pipe ID"].isin(pipes["Pipe ID"]),
    "Pipe ID"
]

print("Leak records with Pipe IDs not found in pipes:", len(unmatched))

if len(unmatched) > 0:
    display(unmatched.head())

Leak records with Pipe IDs not found in pipes: 0


## 3. Check Missing Pipe Information

The `pipes` dataset contains:

- 15 missing installation dates
- 8 missing material values

Before removing or imputing these observations, we examine whether these pipes also appear in the historical leak data.

This allows us to define a consistent missing-value rule and align our preprocessing with the Cost Prediction team where applicable.

In [14]:
# Pipes with missing lay date
missing_lay = pipes[pipes["Lay date"].isna()]

# Pipes with missing material
missing_material = pipes[pipes["Material"].isna()]

print("Pipes with missing Lay date:", len(missing_lay))
print("Pipes with missing Material:", len(missing_material))

print("\nMissing Lay date pipes:")
display(missing_lay)

print("\nMissing Material pipes:")
display(missing_material)

Pipes with missing Lay date: 15
Pipes with missing Material: 8

Missing Lay date pipes:


,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2
9868,P10869,NaN,wrought iron,water,998.44,4234.71,1004.06,4237.59
9869,P10870,NaN,wrought iron,water,998.44,4234.71,994.69,4236.88
9870,P10871,NaN,wrought iron,water,1087.96,4280.97,1087.76,4276.32
9871,P10872,NaN,wrought iron,grassland,1087.96,4280.97,1089.18,4285.56
9872,P10873,NaN,wrought iron,water,1087.96,4280.97,1082.37,4278.07
9873,P10874,NaN,wrought iron,grassland,1126.90,4427.72,1128.79,4434.31
9874,P10875,NaN,wrought iron,grassland,1125.68,4423.15,1128.61,4434.70
11274,P12275,NaN,wrought iron,grassland,2588.77,3203.34,2591.18,3209.41
11275,P12276,NaN,wrought iron,grassland,2590.64,3211.32,2587.31,3197.17
11276,P12277,NaN,wrought iron,grassland,2492.25,2768.58,2488.03,2761.18



Missing Material pipes:


,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2
8985,P09986,1962-02-20,NaN,water,584.00,2499.90,577.27,2502.69
8986,P09987,1947-11-02,NaN,water,595.04,2497.15,606.16,2490.79
41035,P42036,1962-03-17,NaN,water,435.63,2561.51,426.31,2565.49
41036,P42037,1962-03-13,NaN,water,454.35,2553.57,444.99,2557.55
41037,P42038,1962-03-10,NaN,water,472.05,2546.24,463.36,2549.80
41038,P42039,1962-02-23,NaN,water,523.49,2524.97,536.92,2519.38
41039,P42040,1962-02-22,NaN,water,550.39,2513.82,563.84,2508.25
41040,P42041,1962-02-21,NaN,water,563.83,2508.26,577.27,2502.68


In [15]:
# Check whether pipes with missing information appear in costs
missing_lay_with_leak = missing_lay[
    missing_lay["Pipe ID"].isin(costs["Pipe ID"])
]

missing_material_with_leak = missing_material[
    missing_material["Pipe ID"].isin(costs["Pipe ID"])
]

print("Missing Lay date AND has leak record:",
      len(missing_lay_with_leak))

print("Missing Material AND has leak record:",
      len(missing_material_with_leak))

print("\nMissing Lay date with leak:")
display(missing_lay_with_leak)

print("\nMissing Material with leak:")
display(missing_material_with_leak)

Missing Lay date AND has leak record: 6
Missing Material AND has leak record: 0

Missing Lay date with leak:


,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2
9869,P10870,NaN,wrought iron,water,998.44,4234.71,994.69,4236.88
9870,P10871,NaN,wrought iron,water,1087.96,4280.97,1087.76,4276.32
9871,P10872,NaN,wrought iron,grassland,1087.96,4280.97,1089.18,4285.56
9872,P10873,NaN,wrought iron,water,1087.96,4280.97,1082.37,4278.07
9873,P10874,NaN,wrought iron,grassland,1126.90,4427.72,1128.79,4434.31
9874,P10875,NaN,wrought iron,grassland,1125.68,4423.15,1128.61,4434.70



Missing Material with leak:


,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2


## 4. Remove Pipes with Missing Baseline Information

Our baseline leak prediction model requires both pipe age and material.

- 15 pipes have missing installation dates, so pipe age cannot be calculated.
- 8 pipes have missing material information.
- There is no overlap between these missing-value groups.

Because only 23 out of 43,039 pipes are affected, these pipes are excluded from the baseline modeling dataset rather than imputed.

This is also consistent with the Cost Prediction team's treatment of leak records with missing installation dates.

In [16]:
# Remove pipes missing information required for baseline features
pipes_clean = pipes.dropna(
    subset=["Lay date", "Material"]
).copy()

print("Original number of pipes:", len(pipes))
print("Pipes removed:", len(pipes) - len(pipes_clean))
print("Remaining pipes:", len(pipes_clean))

print("\nRemaining missing values:")
display(pipes_clean.isna().sum())

Original number of pipes: 43039
Pipes removed: 23
Remaining pipes: 43016

Remaining missing values:


Pipe ID     0
Lay date    0
Material    0
Surface     0
GPS x1      0
GPS y1      0
GPS x2      0
GPS y2      0
dtype: int64

## 5. Create Basic Pipe Features

The installation date and leak date are converted to datetime format.

Pipe length is calculated using the same definition used by the Cost Prediction team:

\[
\text{Pipe Length}
=
\sqrt{(x_2-x_1)^2 + (y_2-y_1)^2}
\]

Using the same definition ensures consistency between the leak prediction and cost prediction components of the project.

In [17]:
# Convert dates to datetime
pipes_clean["Lay date"] = pd.to_datetime(pipes_clean["Lay date"])
costs["Date"] = pd.to_datetime(costs["Date"])

# Calculate pipe length using the same definition as the Cost Prediction team
pipes_clean["Pipe length"] = np.sqrt(
    (pipes_clean["GPS x2"] - pipes_clean["GPS x1"])**2 +
    (pipes_clean["GPS y2"] - pipes_clean["GPS y1"])**2
)

# Extract leak year
costs["Leak year"] = costs["Date"].dt.year

display(pipes_clean.head())
display(costs.head())

,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2,Pipe length
0,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,12.140000
1,P01002,1971-09-19,cast iron,water,6512.35,4147.58,6536.59,4201.94,59.519637
2,P01003,1971-09-18,cast iron,water,6536.59,4201.94,6518.42,4201.93,18.170003
3,P01004,1982-05-22,cast iron,structure,2836.68,3377.95,2848.80,3405.12,29.750686
4,P01005,1971-07-03,cast iron,water,5117.28,4272.91,5126.37,4297.16,25.897695


,Pipe ID,Date,Time,Cost,Leak year
0,P19711,2019-01-06,08:20,2074.90,2019
1,P07628,2019-01-07,00:00,618.71,2019
2,P31195,2019-04-16,03:48,1050.07,2019
3,P15715,2019-04-16,16:51,1000.81,2019
4,P10415,2019-04-17,22:08,3157.83,2019


In [18]:
print("Lay date type:", pipes_clean["Lay date"].dtype)
print("Leak date type:", costs["Date"].dtype)

print("\nLeak records by year:")
display(costs["Leak year"].value_counts().sort_index())

Lay date type: datetime64[ns]
Leak date type: datetime64[ns]

Leak records by year:


Leak year
2019    1078
2020     703
2021     866
2022    2874
2023    1826
2024    1955
2025    1773
2026    3038
Name: count, dtype: int64

## 6. Merge Leak Information with Pipe Data

Historical leak information is merged with the cleaned pipe dataset using `Pipe ID`.

Pipes without a recorded leak between 2019 and 2026 are retained. Their `Leak year` will remain missing, indicating that no leak was observed during the historical observation period.

Only the leak year is needed to construct the initial pipe-year target variable.

In [19]:
# Keep only the leak information needed for the initial dataset
leak_info = costs[["Pipe ID", "Date", "Leak year"]].copy()

# Merge leak information onto all cleaned pipes
pipes_merged = pipes_clean.merge(
    leak_info,
    on="Pipe ID",
    how="left"
)

print("Clean pipes:", len(pipes_clean))
print("Merged rows:", len(pipes_merged))
print("Pipes with recorded leak:", pipes_merged["Leak year"].notna().sum())
print("Pipes without recorded leak:", pipes_merged["Leak year"].isna().sum())

display(pipes_merged.head())

Clean pipes: 43016
Merged rows: 43016
Pipes with recorded leak: 14107
Pipes without recorded leak: 28909


,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2,Pipe length,Date,Leak year
0,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,12.140000,NaT,NaN
1,P01002,1971-09-19,cast iron,water,6512.35,4147.58,6536.59,4201.94,59.519637,2023-06-21,2023.0
2,P01003,1971-09-18,cast iron,water,6536.59,4201.94,6518.42,4201.93,18.170003,2024-04-28,2024.0
3,P01004,1982-05-22,cast iron,structure,2836.68,3377.95,2848.80,3405.12,29.750686,NaT,NaN
4,P01005,1971-07-03,cast iron,water,5117.28,4272.91,5126.37,4297.16,25.897695,2024-07-03,2024.0


## 7. Create Pipe-Year Observations

Each pipe is expanded into annual observations from 2019 through 2026.

For each pipe-year:

- `Y = 0` if the pipe has not leaked by that year.
- `Y = 1` in the year of its first recorded leak.
- Observations after the first leak are excluded because the original pipe is assumed to have been replaced.

Therefore, each pipe can contribute at most one positive (`Y = 1`) observation.

In [20]:
# Create one row per pipe per year
years = pd.DataFrame({"Year": range(2019, 2027)})

pipe_year = (
    pipes_merged
    .merge(years, how="cross")
)

# Keep observations only up to and including the leak year
pipe_year = pipe_year[
    pipe_year["Leak year"].isna() |
    (pipe_year["Year"] <= pipe_year["Leak year"])
].copy()

# Create target variable
pipe_year["Y"] = (
    pipe_year["Year"] == pipe_year["Leak year"]
).astype(int)

print("Total pipe-year observations:", len(pipe_year))
print("Leak observations (Y=1):", pipe_year["Y"].sum())
print("Non-leak observations (Y=0):", (pipe_year["Y"] == 0).sum())

display(pipe_year.head(10))

Total pipe-year observations: 305391
Leak observations (Y=1): 14107
Non-leak observations (Y=0): 291284


,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2,Pipe length,Date,Leak year,Year,Y
0,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,12.140000,NaT,NaN,2019,0
1,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,12.140000,NaT,NaN,2020,0
2,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,12.140000,NaT,NaN,2021,0
3,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,12.140000,NaT,NaN,2022,0
4,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,12.140000,NaT,NaN,2023,0
5,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,12.140000,NaT,NaN,2024,0
6,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,12.140000,NaT,NaN,2025,0
7,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,12.140000,NaT,NaN,2026,0
8,P01002,1971-09-19,cast iron,water,6512.35,4147.58,6536.59,4201.94,59.519637,2023-06-21,2023.0,2019,0
9,P01002,1971-09-19,cast iron,water,6512.35,4147.58,6536.59,4201.94,59.519637,2023-06-21,2023.0,2020,0


## 8. Calculate Pipe Age for Each Pipe-Year

Pipe age must change across yearly observations.

For the leak prediction dataset, age is defined as the pipe's age at the beginning of each observation year (January 1).

Age is calculated using elapsed days divided by 365.25, consistent with the Cost Prediction team's general age calculation approach.

In [21]:
# Reference date = January 1 of each observation year
pipe_year["Year start"] = pd.to_datetime(
    pipe_year["Year"].astype(str) + "-01-01"
)

# Calculate pipe age at the beginning of each year
pipe_year["Pipe age"] = (
    (pipe_year["Year start"] - pipe_year["Lay date"]).dt.days
    / 365.25
)

display(
    pipe_year[
        ["Pipe ID", "Year", "Lay date", "Pipe age",
         "Leak year", "Y"]
    ].head(10)
)

,Pipe ID,Year,Lay date,Pipe age,Leak year,Y
0,P01001,2019,1955-03-11,63.811088,NaN,0
1,P01001,2020,1955-03-11,64.810404,NaN,0
2,P01001,2021,1955-03-11,65.812457,NaN,0
3,P01001,2022,1955-03-11,66.811773,NaN,0
4,P01001,2023,1955-03-11,67.811088,NaN,0
5,P01001,2024,1955-03-11,68.810404,NaN,0
6,P01001,2025,1955-03-11,69.812457,NaN,0
7,P01001,2026,1955-03-11,70.811773,NaN,0
8,P01002,2019,1971-09-19,47.285421,2023.0,0
9,P01002,2020,1971-09-19,48.284736,2023.0,0


## 9. Validate the Pipe-Year Dataset

Before saving the master dataset, we verify that the pipe-year construction follows the intended observation definition.

We check that:

- Each Pipe ID and year combination is unique.
- Each pipe has at most one positive leak observation.
- No observations exist after a pipe's first leak.
- Every retained historical leak corresponds to exactly one `Y = 1`.
- The yearly number of pipes at risk and leak rate are reasonable.

In [22]:
# 1. Check duplicate pipe-year observations
duplicates = pipe_year.duplicated(
    subset=["Pipe ID", "Year"]
).sum()

# 2. Check whether any pipe has more than one Y=1
positive_per_pipe = pipe_year.groupby("Pipe ID")["Y"].sum()
multiple_leaks = (positive_per_pipe > 1).sum()

# 3. Check for observations after first leak
after_leak = pipe_year[
    pipe_year["Leak year"].notna() &
    (pipe_year["Year"] > pipe_year["Leak year"])
]

# 4. Check total positive observations
expected_leaks = pipes_merged["Leak year"].notna().sum()
actual_leaks = pipe_year["Y"].sum()

print("Duplicate pipe-year rows:", duplicates)
print("Pipes with more than one Y=1:", multiple_leaks)
print("Observations after first leak:", len(after_leak))
print("Expected leak observations:", expected_leaks)
print("Actual Y=1 observations:", actual_leaks)

Duplicate pipe-year rows: 0
Pipes with more than one Y=1: 0
Observations after first leak: 0
Expected leak observations: 14107
Actual Y=1 observations: 14107


In [23]:
year_summary = (
    pipe_year
    .groupby("Year")
    .agg(
        pipes_at_risk=("Pipe ID", "count"),
        leaks=("Y", "sum")
    )
)

year_summary["non_leaks"] = (
    year_summary["pipes_at_risk"] - year_summary["leaks"]
)

year_summary["leak_rate"] = (
    year_summary["leaks"] / year_summary["pipes_at_risk"]
)

display(year_summary)

,pipes_at_risk,leaks,non_leaks,leak_rate
Year,,,,
2019,43016,1078,41938,0.025060
2020,41938,703,41235,0.016763
2021,41235,866,40369,0.021002
2022,40369,2874,37495,0.071193
2023,37495,1824,35671,0.048646
2024,35671,1951,33720,0.054694
2025,33720,1773,31947,0.052580
2026,31947,3038,28909,0.095095


In [24]:
display(
    pipe_year.loc[
        pipe_year["Pipe ID"] == "P01002",
        ["Pipe ID", "Year", "Pipe age", "Leak year", "Y"]
    ]
)

,Pipe ID,Year,Pipe age,Leak year,Y
8,P01002,2019,47.285421,2023.0,0
9,P01002,2020,48.284736,2023.0,0
10,P01002,2021,49.286790,2023.0,0
11,P01002,2022,50.286105,2023.0,0
12,P01002,2023,51.285421,2023.0,1


## 10. Create the Master Leak Prediction Dataset

The validated pipe-year data is reduced to the variables needed for leak prediction.

The initial master dataset contains:

- `pipe_id`: pipe identifier
- `year`: observation year
- `material`: pipe material
- `surface`: surface type
- `pipe_length`: calculated pipe length
- `pipe_age`: pipe age at the beginning of the observation year
- `Y`: whether the pipe leaks during that year

Additional features can be added later during the modeling process.

The master dataset contains all years from 2019–2026. Train-test splitting will be performed later during modeling using expanding-window validation.

In [25]:
master = pipe_year[
    [
        "Pipe ID",
        "Year",
        "Material",
        "Surface",
        "Pipe length",
        "Pipe age",
        "Y"
    ]
].copy()

# Standardize column names
master.columns = [
    "pipe_id",
    "year",
    "material",
    "surface",
    "pipe_length",
    "pipe_age",
    "Y"
]

# Sort for readability
master = master.sort_values(
    ["pipe_id", "year"]
).reset_index(drop=True)

print("Master dataset shape:", master.shape)

display(master.head(10))

Master dataset shape: (305391, 7)


,pipe_id,year,material,surface,pipe_length,pipe_age,Y
0,P01001,2019,copper,grassland,12.140000,63.811088,0
1,P01001,2020,copper,grassland,12.140000,64.810404,0
2,P01001,2021,copper,grassland,12.140000,65.812457,0
3,P01001,2022,copper,grassland,12.140000,66.811773,0
4,P01001,2023,copper,grassland,12.140000,67.811088,0
5,P01001,2024,copper,grassland,12.140000,68.810404,0
6,P01001,2025,copper,grassland,12.140000,69.812457,0
7,P01001,2026,copper,grassland,12.140000,70.811773,0
8,P01002,2019,cast iron,water,59.519637,47.285421,0
9,P01002,2020,cast iron,water,59.519637,48.284736,0


## 11. Final Validation

A final check is performed before saving the dataset to ensure that the master modeling data contains no missing values or duplicate pipe-year observations.

In [26]:
print("Shape:", master.shape)

print("\nMissing values:")
display(master.isna().sum())

print(
    "\nDuplicate pipe-year observations:",
    master.duplicated(["pipe_id", "year"]).sum()
)

print("\nTarget distribution:")
display(master["Y"].value_counts())

print("\nTarget proportion:")
display(master["Y"].value_counts(normalize=True))

Shape: (305391, 7)

Missing values:


pipe_id        0
year           0
material       0
surface        0
pipe_length    0
pipe_age       0
Y              0
dtype: int64


Duplicate pipe-year observations: 0

Target distribution:


Y
0    291284
1     14107
Name: count, dtype: int64


Target proportion:


Y
0    0.953807
1    0.046193
Name: proportion, dtype: float64

## 12. Save the Master Dataset

The completed pipe-year dataset is saved as a shared master file.

All leak prediction models should use this same dataset so that model comparisons are based on identical observations and feature definitions.

In [27]:
master.to_csv(
    "../data/leak_prediction_master.csv",
    index=False
)

print("Saved: ../data/leak_prediction_master.csv")

Saved: ../data/leak_prediction_master.csv
